# Block 2 — Clinical Knowledge Graph & Prior Validation Engine
**Medical Document Intelligence System (Batch Pipeline)**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RwaRwa599/epq3/blob/block2/block2/Block_2_Medical_Knowledge_Graph.ipynb)

---
### End-to-End Batch Pipeline Overview
1. **Ingest Block 1 Output ZIP:** Receives `block1_normalized_batch.zip` via interactive file upload popup.
2. **Health Profile Expansion:** Expands composite plans (`Lipid`, `Liver`, `Renal`, `Diabetes`, `Anemia`) into component tests.
3. **Specimen Tube Rule Enforcement:** Computes exact container constraints (`EDTA`, `CB`, `Fl`, `Cit`, `Urine`, `Stool`, `pap`, `UBT`).
4. **Bayesian Prior Ranking (`assume()`):** Biases handwriting interpretations based on marked tests.
5. **Cross-Field Clinical Validation:** Flags tube shortages, redundant orders, and generates audit reports.
6. **Export Block 3 ZIP (`block2_validated_batch.zip`):** Bundles all canonical images, handwriting crops, prior rankings, and validation reports ready for **Block 3 (HTR)**.

## 1. Setup & Environment
Install minimal requirements (`pydantic`, `matplotlib`) and import `med_doc.kg`.

In [ ]:
import os, sys, glob, json, zipfile, shutil

# In Google Colab: Ensure latest repo code is cloned/updated
if 'google.colab' in sys.modules or os.path.exists('/content'):
    repo_dir = '/content/repo'
    if os.path.exists(repo_dir):
        # Pull latest commit from block2 branch
        !cd /content/repo && git fetch origin block2 && git reset --hard origin/block2
    else:
        # Fresh clone
        !git clone -b block2 https://github.com/RwaRwa599/epq3.git /content/repo

    block2_path = '/content/repo/block2'
    if block2_path not in sys.path:
        sys.path.insert(0, block2_path)
    os.chdir(block2_path)

# Clear any cached med_doc imports from prior runs in this kernel
for mod in list(sys.modules.keys()):
    if mod.startswith('med_doc'):
        del sys.modules[mod]

!pip install -q "pydantic>=2.0.0" "matplotlib>=3.7.0"

from med_doc.kg import KnowledgeGraph, process_batch_from_block1
print("✓ Environment initialized and med_doc.kg imported successfully!")

## 2. Upload Block 1 Batch ZIP (`block1_normalized_batch.zip`)
Upload the ZIP file generated by **Block 1**. The file upload popup will prompt you to choose the ZIP.

In [ ]:
input_zip = "block1_normalized_batch.zip"

try:
    from google.colab import files
    print("Upload the 'block1_normalized_batch.zip' output file from Block 1:")
    uploaded = files.upload()
    for filename in uploaded.keys():
        if filename.endswith(".zip"):
            input_zip = filename
            print(f"[+] Ingested ZIP: {input_zip}")
            break
except Exception as e:
    print("Interactive upload popup skipped or running locally.")

# If no uploaded zip, generate a synthetic mock batch for instant testing
if not os.path.exists(input_zip):
    print("[!] No uploaded ZIP found. Creating synthetic demo mock Block 1 batch...")
    os.makedirs("mock_block1/docs/sample_sheet_01/crops/handwriting", exist_ok=True)
    os.makedirs("mock_block1/docs/sample_sheet_01/crops/checkboxes", exist_ok=True)
    mock_meta = {
        "doc_id": "sample_sheet_01",
        "template_id": "v1",
        "canvas_size": [2048, 1754],
        "alignment_confidence": 0.98,
        "num_checkboxes": 138,
        "num_handwriting": 12,
        "detected_marks": {
            "cbc": {"dark_ratio": 0.28, "is_marked_candidate": True},
            "profile_lipid": {"dark_ratio": 0.32, "is_marked_candidate": True},
            "glucose_fasting": {"dark_ratio": 0.24, "is_marked_candidate": True}
        },
        "fields": {
            "handwriting": {"others": {"bbox": [0.5, 0.8, 0.9, 0.85]}}
        }
    }
    with open("mock_block1/docs/sample_sheet_01/metadata.json", "w") as f:
        json.dump(mock_meta, f, indent=2)
    with open("mock_block1/manifest.json", "w") as f:
        json.dump({"version": "1.0", "block": "block1", "total_documents": 1, "documents": [{"doc_id": "sample_sheet_01", "status": "success"}]}, f, indent=2)
    with zipfile.ZipFile(input_zip, "w") as zf:
        for root, _, files_list in os.walk("mock_block1"):
            for f_name in files_list:
                f_p = os.path.join(root, f_name)
                zf.write(f_p, os.path.relpath(f_p, "mock_block1"))
    print(f"[+] Created demo {input_zip}")

print(f"\n✓ Ready to process: '{input_zip}' ({os.path.getsize(input_zip)/1024:.1f} KB)")

## 3. Run Block 2 Batch Validation & Prior Engine
Ingests all documents from the Block 1 ZIP, applies clinical Knowledge Graph rules, resolves priors, and generates validation reports.

In [ ]:
output_zip_path = "block2_validated_batch.zip"
output_dir = "outputs/block2_batch"

result = process_batch_from_block1(
    input_source=input_zip,
    output_dir=output_dir,
    output_zip=output_zip_path
)

manifest = result["manifest"]
print("\n" + "="*70)
print(f"[+] Batch Processing Finished!")
print(f"    Total Sheets:   {manifest['total_documents']}")
print(f"    Valid Orders:   {manifest['valid_documents']}")
print("="*70)

## 4. Inspect Document Validation Reports & Prior Rankings
Inspect the structured clinical report, tube requirements, and Bayesian handwriting prior rankings for each sheet.

In [ ]:
for doc in manifest["documents"]:
    doc_id = doc["doc_id"]
    print(f"\n================ DOCUMENT: {doc_id} ================")
    print(f"  • Status:        {'✓ VALID' if doc['is_valid'] else '✗ DISCREPANCY'}")
    print(f"  • Confidence:    {doc['confidence']:.2f}")
    print(f"  • Ticked Tests:  {doc['ticked_tests']}")
    print(f"  • Implied Tests: {doc['implied_tests']}")
    print(f"  • Tube Demand:   {doc['expected_tubes']}")
    
    report_path = f"{output_dir}/{doc['validation_report_path']}"
    if os.path.exists(report_path):
        with open(report_path) as f:
            rep = json.load(f)
            if rep.get('discrepancies'):
                print(f"  • Discrepancies: {rep['discrepancies']}")
            if rep.get('warnings'):
                print(f"  • Warnings:      {rep['warnings']}")
    
    priors_path = f"{output_dir}/{doc['prior_rankings_path']}"
    if os.path.exists(priors_path):
        with open(priors_path) as f:
            priors = json.load(f)
            print(f"  • Prior Rankings (Handwriting): {list(priors.keys())}")

## 5. Download `block2_validated_batch.zip` for Block 3
Download the validated batch archive packaged specifically for **Block 3 (Handwriting Recognition & HTR)**.

In [ ]:
try:
    from google.colab import files
    print(f"Downloading {output_zip_path} for Block 3...")
    files.download(output_zip_path)
except Exception as e:
    print(f"ZIP file ready locally at: {os.path.abspath(output_zip_path)}")